# Full Fine-Tuning: Training-Label Noise Experiments

**MIA5150 — Full BERT fine-tuning**  
**Team:** Callum Anderson and Zaynab Othman Taha

Supervised fine-tuning for 11-class paper classification: train all BERT encoder weights and the same linear classification head used in frozen BERT. Each seed/noise condition starts from the original pretrained checkpoint.

BERT → 768-dimensional [CLS] vector → linear classifier → 11 logits → softmax probabilities

## Imports


In [2]:
import gc
import hashlib
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import transformers
from transformers import AutoModel, AutoTokenizer, BertConfig, set_seed

from IPython.display import display


# 1. Verify GPU, as it will be needed for this experiment

In [3]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


# 2. Load the prepared data and set the experiment conditions

The conditions are **0%, 10%, 20%, and 40% added training-label noise**, with
**seeds 42, 43, and 44**. 

In [4]:
LABEL_COLUMN = "label (target)"
NUMBER_OF_CLASSES = 11
CLASS_IDS = np.arange(NUMBER_OF_CLASSES)
NOISE_PERCENTAGES = (0, 10, 20, 40)
RANDOM_SEEDS = (42, 43, 44)
SPLIT_NAMES = ("train", "validation", "test")

DATA_DIRECTORY = Path("/content")
source_paths = {split: DATA_DIRECTORY / f"{split}_merged.csv" for split in SPLIT_NAMES}
missing_files = [path.name for path in source_paths.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing files in /content: {', '.join(missing_files)}. "
        "Upload the three cleaned CSVs directly to the Colab Files panel."
    )

source_hashes = {
    split: hashlib.sha256(path.read_bytes()).hexdigest()
    for split, path in source_paths.items()
}
data_splits = {
    split: pd.read_csv(path, dtype={"paper_id": "string"})
    for split, path in source_paths.items()
}
for split, frame in data_splits.items():
    assert frame["paper_id"].notna().all() and frame["paper_id"].is_unique, split
    assert pd.api.types.is_integer_dtype(frame[LABEL_COLUMN]), split
    assert set(frame[LABEL_COLUMN]) == set(CLASS_IDS), split
    for text_column in ("title", "abstract"):
        assert frame[text_column].fillna("").str.strip().ne("").all(), split

for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert set(data_splits[left]["paper_id"]).isdisjoint(data_splits[right]["paper_id"])

training_data = data_splits["train"]
validation_data = data_splits["validation"]
test_data = data_splits["test"]
original_training_labels = training_data[LABEL_COLUMN].to_numpy(dtype=np.int64, copy=True)
original_training_labels.setflags(write=False)
original_split_labels = {split: frame[LABEL_COLUMN].copy(deep=True) for split, frame in data_splits.items()}
class_names = training_data.groupby(LABEL_COLUMN)["classname"].first().reindex(CLASS_IDS)
for frame in data_splits.values():
    assert frame["classname"].eq(frame[LABEL_COLUMN].map(class_names)).all()

print(f"Loaded {sum(len(frame) for frame in data_splits.values()):,} unique papers from {DATA_DIRECTORY}.")


Loaded 23,335 unique papers from /content.


# 3. Add symmetric training-label noise

For each condition, select records **proportionally from each original training class**
and replace each selected label uniformly with **one of the other ten classes**.
Every selected label therefore changes. Validation and test labels stay unchanged.


In [5]:
def add_symmetric_label_noise(original_labels, noise_percent, seed):
    """Return independent noisy labels and a mask of changed records."""
    labels = np.asarray(original_labels)
    if labels.ndim != 1 or labels.size == 0 or not np.issubdtype(labels.dtype, np.integer):
        raise ValueError("Labels must be a nonempty one-dimensional integer array.")
    if not np.isin(labels, CLASS_IDS).all():
        raise ValueError("Labels must be integers from 0 to 10.")
    if not isinstance(noise_percent, (int, np.integer)) or not 0 <= noise_percent <= 100:
        raise ValueError("noise_percent must be an integer between 0 and 100.")
    if not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError("seed must be a nonnegative integer.")

    noisy_labels = labels.copy()
    changed_mask = np.zeros(len(labels), dtype=bool)
    original_class_counts = np.bincount(labels, minlength=NUMBER_OF_CLASSES)
    quota_numerators = original_class_counts * noise_percent
    flips_per_class = quota_numerators // 100
    target_flips = (len(labels) * noise_percent + 50) // 100
    remaining = int(target_flips - flips_per_class.sum())
    remainder_order = np.argsort(-(quota_numerators % 100), kind="stable")
    flips_per_class[remainder_order[:remaining]] += 1

    rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(noise_percent)]))
    for class_id, number_to_flip in enumerate(flips_per_class):
        if number_to_flip == 0:
            continue
        eligible_positions = np.flatnonzero(labels == class_id)
        chosen_positions = rng.choice(eligible_positions, size=int(number_to_flip), replace=False)
        alternative_labels = CLASS_IDS[CLASS_IDS != class_id]
        noisy_labels[chosen_positions] = rng.choice(alternative_labels, size=int(number_to_flip))
        changed_mask[chosen_positions] = True

    assert changed_mask.sum() == target_flips
    assert np.array_equal(changed_mask, noisy_labels != labels)
    return noisy_labels, changed_mask


## 3.1 Generate the reusable label variants

`training_label_variants[(seed, noise_percent)]` holds the labels aligned with
`training_data`. That same entry is used for frozen BERT, LoRA, and full fine-tuning.



In [6]:
training_label_variants = {}
noise_audits = {}

for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        noisy_labels, changed_mask = add_symmetric_label_noise(
            original_training_labels, noise_percent, seed
        )
        noisy_labels.setflags(write=False)
        key = (seed, noise_percent)
        training_label_variants[key] = noisy_labels
        noise_audits[key] = pd.DataFrame({
            "paper_id": training_data["paper_id"].to_numpy(copy=True),
            "original_label": original_training_labels.copy(),
            "noisy_label": noisy_labels.copy(),
            "label_changed": changed_mask.copy(),
        })


## 3.2 Verify the scheme

Check exact flip totals, proportional allocation, valid replacement labels,
reproducibility, and alignment to paper IDs. Also confirm the original labels and
all three source files are unchanged. These checks do not establish that the
original disease labels are semantically correct.

In [7]:
original_class_counts = np.bincount(original_training_labels, minlength=NUMBER_OF_CLASSES)
for (seed, noise_percent), noisy_labels in training_label_variants.items():
    audit = noise_audits[(seed, noise_percent)]
    changed = noisy_labels != original_training_labels
    expected_flips = (len(training_data) * noise_percent + 50) // 100
    assert int(changed.sum()) == expected_flips
    assert np.isin(noisy_labels, CLASS_IDS).all()
    assert not np.shares_memory(noisy_labels, original_training_labels)
    assert np.array_equal(audit["paper_id"], training_data["paper_id"])
    assert np.array_equal(audit["original_label"], original_training_labels)
    assert np.array_equal(audit["noisy_label"], noisy_labels)
    assert np.array_equal(audit["label_changed"], changed)

    actual_per_class = np.bincount(original_training_labels[changed], minlength=NUMBER_OF_CLASSES)
    lower_quota = original_class_counts * noise_percent // 100
    upper_quota = (original_class_counts * noise_percent + 99) // 100
    assert ((actual_per_class >= lower_quota) & (actual_per_class <= upper_quota)).all()
    repeated_labels, repeated_mask = add_symmetric_label_noise(
        original_training_labels, noise_percent, seed
    )
    assert np.array_equal(noisy_labels, repeated_labels)
    assert np.array_equal(changed, repeated_mask)

for split, frame in data_splits.items():
    pd.testing.assert_series_equal(frame[LABEL_COLUMN], original_split_labels[split])
    assert hashlib.sha256(source_paths[split].read_bytes()).hexdigest() == source_hashes[split]
assert np.array_equal(training_data[LABEL_COLUMN], original_training_labels)
print("Passed: all 12 variants are reproducible, correctly allocated, and aligned.")
print("Original training, validation, test labels and CSV files remain unchanged.")


Passed: all 12 variants are reproducible, correctly allocated, and aligned.
Original training, validation, test labels and CSV files remain unchanged.


# 4. Tokenize titles and abstracts

BERT-base-uncased tokenizer and 256-token limit.


In [8]:
BERT_CHECKPOINT_NAME = "google-bert/bert-base-uncased"
MAXIMUM_TOKEN_LENGTH = 256

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT_NAME, use_fast=True)


def tokenize_papers(paper_data):
    return bert_tokenizer(
        text=paper_data["title"].tolist(),
        text_pair=paper_data["abstract"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAXIMUM_TOKEN_LENGTH,
        return_tensors="pt",
    )


tokenized_splits = {
    split: tokenize_papers(data_splits[split])
    for split in SPLIT_NAMES
}
training_tokenized_inputs = tokenized_splits["train"]
validation_tokenized_inputs = tokenized_splits["validation"]
test_tokenized_inputs = tokenized_splits["test"]

print("Tokenizer:", BERT_CHECKPOINT_NAME)
print("Maximum sequence length:", MAXIMUM_TOKEN_LENGTH)
for split, encoded_inputs in tokenized_splits.items():
    expected_shape = (len(data_splits[split]), MAXIMUM_TOKEN_LENGTH)
    for input_name in ("input_ids", "attention_mask", "token_type_ids"):
        assert tuple(encoded_inputs[input_name].shape) == expected_shape, (split, input_name)
    print(f"{split}: {expected_shape[0]:,} records × {expected_shape[1]} tokens")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer: google-bert/bert-base-uncased
Maximum sequence length: 256
train: 16,334 records × 256 tokens
validation: 3,499 records × 256 tokens
test: 3,502 records × 256 tokens


# 5. Prepare BERT for full fine-tuning

Use the same **google-bert/bert-base-uncased** encoder and raw **[CLS] → linear 11-class classifier** as frozen. Omit the unused BERT pooler because this classifier never reads it.

Tokens pass through BERT on every training batch. All encoder and head parameters receive gradients. BERT's standard dropout is active during training and disabled during evaluation; there is no additional classifier dropout. Construct a new model for each run so learned weights are never carried into another noise level.


In [9]:
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime, then rerun the notebook before training.")
computation_device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(computation_device))
TOKEN_INPUT_NAMES = ("input_ids", "attention_mask", "token_type_ids")

class FullBertClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.bert = encoder
        self.classifier = nn.Linear(encoder.config.hidden_size, NUMBER_OF_CLASSES)

    def forward(self, **tokenized_inputs):
        encoded = self.bert(**tokenized_inputs)
        return self.classifier(encoded.last_hidden_state[:, 0, :])

def create_full_model(seed):
    encoder = AutoModel.from_pretrained(BERT_CHECKPOINT_NAME, add_pooling_layer=False)
    # Match frozen's seeded nn.Linear initialization after loading the encoder.
    set_seed(seed)
    model = FullBertClassifier(encoder).float().to(computation_device)
    model.requires_grad_(True)
    assert all(parameter.requires_grad for parameter in model.parameters())
    return model

def synchronize_gpu():
    if computation_device.type == "cuda":
        torch.cuda.synchronize(computation_device)

def reset_gpu_peak_memory():
    if computation_device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(computation_device)

def peak_gpu_memory_gib():
    return (torch.cuda.max_memory_allocated(computation_device) / 1024 ** 3
            if computation_device.type == "cuda" else np.nan)

def maximum_measured_memory(*values):
    measured = [float(value) for value in values if np.isfinite(value)]
    return max(measured) if measured else np.nan

def make_batches(split, labels, batch_size, shuffle=False, generator=None):
    inputs = tokenized_splits[split]
    dataset = TensorDataset(*(inputs[name] for name in TOKEN_INPUT_NAMES),
                            torch.tensor(np.asarray(labels).copy(), dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      pin_memory=computation_device.type == "cuda", generator=generator)

def unpack_batch(batch):
    inputs = {name: values.to(computation_device, non_blocking=True)
              for name, values in zip(TOKEN_INPUT_NAMES, batch[:-1])}
    return inputs, batch[-1].to(computation_device, non_blocking=True)

@torch.no_grad()
def evaluate_full_model(model, split, true_labels):
    model.eval()
    batches = make_batches(split, true_labels, EVALUATION_BATCH_SIZE,
                           generator=torch.Generator().manual_seed(0))
    predictions = []
    for batch in batches:
        inputs, _ = unpack_batch(batch)
        predictions.append(model(**inputs).argmax(dim=1).cpu())
    predictions = torch.cat(predictions).numpy()
    return {
        "accuracy": float(accuracy_score(true_labels, predictions)),
        "macro_f1": float(f1_score(true_labels, predictions, labels=CLASS_IDS,
                                  average="macro", zero_division=0)),
        "predictions": predictions,
    }

true_validation_labels = validation_data[LABEL_COLUMN].to_numpy(dtype=np.int64, copy=True)
true_test_labels = test_data[LABEL_COLUMN].to_numpy(dtype=np.int64, copy=True)

def restore_full_model(run_key):
    """Rebuild a final model from CPU weights retained in this runtime."""
    metadata = sft_run_metadata[run_key]
    assert metadata["training_run_id"] == sft_training_run_id
    assert metadata["epoch"] == NUMBER_OF_EPOCHS, "Finish all fixed epochs before evaluation."
    assert metadata["noise_label_sha256"] == hashlib.sha256(
        training_label_variants[run_key].astype(np.int64).tobytes()
    ).hexdigest(), "Training labels changed; rerun section 6."
    encoder = AutoModel.from_config(BertConfig.from_dict(metadata["bert_config"]), add_pooling_layer=False)
    model = FullBertClassifier(encoder).float()
    model.load_state_dict(sft_model_states[run_key])
    return model.to(computation_device), metadata


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


# 6. Train BERT and the classification head

Cross-entropy loss, AdamW, weight decay 0.01, batch size 32, and **10 fixed epochs**, as in frozen. The full-fine-tuning learning rate is **0.00002**, fixed across all noise levels and seeds. Frozen uses 0.001 for its small linear head; the proposal fixes hyperparameters within each method. The [original BERT classification example](https://github.com/google-research/bert#sentence-and-sentence-pair-classification-tasks) uses 0.00002 for pretrained BERT updates.

Use ordinary float32 training, without a scheduler or mixed precision. Record validation scores each epoch, but evaluate the **final epoch**, with no early stopping or best-validation selection. Run All processes all 12 conditions sequentially.

The per-epoch history supports later analysis of learning dynamics. [Arpit et al. (2017)](https://proceedings.mlr.press/v70/arpit17a.html) motivate examining how learning and memorization change over training; the logs alone do not establish that noise has been memorized.


In [10]:
NUMBER_OF_EPOCHS = 10
TRAINING_BATCH_SIZE = 32
EVALUATION_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

def current_training_config():
    return {
        "method": "full_finetuning", "checkpoint": BERT_CHECKPOINT_NAME,
        "readout": "raw_cls_linear", "precision": "float32",
        "max_length": MAXIMUM_TOKEN_LENGTH, "source_hashes": dict(source_hashes),
        "class_names": class_names.tolist(), "epochs": NUMBER_OF_EPOCHS,
        "batch_size": TRAINING_BATCH_SIZE, "evaluation_batch_size": EVALUATION_BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "stopping_rule": "final_epoch",
    }

def current_environment():
    return {"torch": str(torch.__version__), "transformers": str(transformers.__version__),
            "gpu": torch.cuda.get_device_name(computation_device) if computation_device.type == "cuda" else "CPU"}

def check_source_data():
    for split in SPLIT_NAMES:
        assert hashlib.sha256(source_paths[split].read_bytes()).hexdigest() == source_hashes[split]
        pd.testing.assert_series_equal(data_splits[split][LABEL_COLUMN], original_split_labels[split])

print(f"Runs: {len(RANDOM_SEEDS) * len(NOISE_PERCENTAGES)}")
print(f"Stopping rule: final epoch ({NUMBER_OF_EPOCHS})")
print(f"Learning rate: {LEARNING_RATE}; training batch size: {TRAINING_BATCH_SIZE}")


Runs: 12
Stopping rule: final epoch (10)
Learning rate: 2e-05; training batch size: 32


## 6.1 Run the 12 training conditions

For each seed/noise pair, start from pretrained BERT, train on its assigned labels, and check the unchanged validation set after each epoch. Backpropagation updates both the encoder and head.

Keep each run's final weights in **CPU memory** for sections 7–8. Training writes no model files, and rerunning this cell trains all 12 conditions again from scratch. Model initialization and copying final weights to CPU are excluded from the recorded training time.


In [11]:
check_source_data()
sft_training_config = current_training_config()
sft_training_run_id = time.time_ns()
sft_model_states = {}
sft_run_metadata = {}
sft_training_history_rows = []

for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        run_key = (seed, noise_percent)
        set_seed(seed)
        model = create_full_model(seed)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        loss_function = nn.CrossEntropyLoss()
        shuffle_generator = torch.Generator().manual_seed(seed)
        training_batches = make_batches("train", training_label_variants[run_key], TRAINING_BATCH_SIZE,
                                        shuffle=True, generator=shuffle_generator)
        history = []
        measurements = {"training_seconds": 0.0, "epoch_validation_seconds": 0.0,
                        "training_and_validation_seconds": 0.0, "peak_pytorch_allocated_gib": np.nan}
        parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\nSeed {seed}, noise {noise_percent}% — {parameter_count:,} trainable parameters")
        synchronize_gpu()
        reset_gpu_peak_memory()
        for epoch in range(1, NUMBER_OF_EPOCHS + 1):
            model.train()
            synchronize_gpu()
            training_start = time.perf_counter()
            total_training_loss = 0.0
            for batch_index, batch in enumerate(tqdm(training_batches, desc=f"Epoch {epoch}/{NUMBER_OF_EPOCHS}", leave=False)):
                inputs, targets = unpack_batch(batch)
                optimizer.zero_grad(set_to_none=True)
                logits = model(**inputs)
                loss = loss_function(logits, targets)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite loss in seed {seed}, noise {noise_percent}%, epoch {epoch}.")
                loss.backward()
                if epoch == 1 and batch_index == 0:
                    assert all(p.grad is not None for p in model.bert.parameters()), "A BERT parameter received no gradient."
                    assert all(p.grad is not None for p in model.classifier.parameters())
                optimizer.step()
                total_training_loss += loss.item() * len(targets)
            del inputs, targets, logits, loss, batch
            synchronize_gpu()
            measurements["training_seconds"] += time.perf_counter() - training_start
            average_loss = total_training_loss / len(training_data)
            validation_start = time.perf_counter()
            validation_metrics = evaluate_full_model(model, "validation", true_validation_labels)
            synchronize_gpu()
            measurements["epoch_validation_seconds"] += time.perf_counter() - validation_start
            measurements["training_and_validation_seconds"] = measurements["training_seconds"] + measurements["epoch_validation_seconds"]
            measurements["peak_pytorch_allocated_gib"] = maximum_measured_memory(
                measurements["peak_pytorch_allocated_gib"], peak_gpu_memory_gib())
            history.append({"seed": seed, "noise_percent": noise_percent, "epoch": epoch,
                            "training_loss": average_loss, "validation_accuracy": validation_metrics["accuracy"],
                            "validation_macro_f1": validation_metrics["macro_f1"]})
            print(f"Epoch {epoch:2d}/{NUMBER_OF_EPOCHS}: loss = {average_loss:.4f}, "
                  f"validation macro-F1 = {validation_metrics['macro_f1']:.4f}")

        sft_model_states[run_key] = {
            name: value.detach().cpu().clone() for name, value in model.state_dict().items()
        }
        sft_run_metadata[run_key] = {
            "training_run_id": sft_training_run_id, "epoch": epoch,
            "noise_label_sha256": hashlib.sha256(training_label_variants[run_key].astype(np.int64).tobytes()).hexdigest(),
            "bert_config": model.bert.config.to_dict(), "measurements": dict(measurements),
            "trainable_parameters": parameter_count, "environment": current_environment(),
        }
        sft_training_history_rows.extend(history)
        print(f"Training + epoch validation: {measurements['training_and_validation_seconds']:.1f} seconds")
        del model, optimizer, training_batches, validation_metrics
        gc.collect()
        if computation_device.type == "cuda":
            torch.cuda.empty_cache()

sft_training_history = pd.DataFrame(sft_training_history_rows)
assert set(sft_model_states) == set(sft_run_metadata) == {(s, n) for s in RANDOM_SEEDS for n in NOISE_PERCENTAGES}
print("\nAll final-epoch weights and epoch histories are held in memory. Continue to validation and final test evaluation.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 42, noise 0% — 108,900,107 trainable parameters


...

Epoch  8/10: loss = 0.6085, validation macro-F1 = 0.6576


Epoch 9/10:   0%|          | 0/511 [00:00<?, ?it/s]

Epoch  9/10: loss = 0.3774, validation macro-F1 = 0.6245


Epoch 10/10:   0%|          | 0/511 [00:00<?, ?it/s]

Epoch 10/10: loss = 0.2373, validation macro-F1 = 0.6265
Training + epoch validation: 555.5 seconds

All final-epoch weights and epoch histories are held in memory. Continue to validation and final test evaluation.


# 7. Evaluate on validation

Rebuild each final-epoch model from its weights held in memory and report individual accuracy and macro-F1 on unchanged validation labels. This does not change training settings or select the best epoch. The test set remains reserved for section 8.


In [12]:
check_source_data()
assert sft_training_config == current_training_config(), "Training settings changed; rerun section 6."
sft_validation_rows = []
sft_classification_reports = {}
sft_validation_predictions = {}
sft_validation_metadata = {}

for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        run_key = (seed, noise_percent)
        synchronize_gpu()
        reset_gpu_peak_memory()
        model, metadata = restore_full_model(run_key)
        synchronize_gpu()
        validation_start = time.perf_counter()
        metrics = evaluate_full_model(model, "validation", true_validation_labels)
        synchronize_gpu()
        validation_seconds = time.perf_counter() - validation_start
        sft_validation_predictions[run_key] = metrics["predictions"]
        sft_classification_reports[run_key] = classification_report(
            true_validation_labels, metrics["predictions"], labels=CLASS_IDS,
            target_names=class_names.tolist(), output_dict=True, zero_division=0)
        sft_validation_metadata[run_key] = {
            "training_run_id": metadata["training_run_id"], "seconds": validation_seconds,
            "peak_pytorch_allocated_gib": peak_gpu_memory_gib(),
        }
        sft_validation_rows.append({"seed": seed, "noise_percent": noise_percent,
                                    "final_epoch": metadata["epoch"], "accuracy": metrics["accuracy"],
                                    "macro_f1": metrics["macro_f1"],
                                    "training_seconds": metadata["measurements"]["training_seconds"]})
        del model
        gc.collect()
        if computation_device.type == "cuda":
            torch.cuda.empty_cache()

sft_validation_results = pd.DataFrame(sft_validation_rows)
print("Final-epoch validation results (accuracy and macro-F1 on a 0–1 scale):")
display(sft_validation_results.round(4))
sft_validation_training_run_id = sft_training_run_id


Final-epoch validation results (accuracy and macro-F1 on a 0–1 scale):


,seed,noise_percent,final_epoch,accuracy,macro_f1,training_seconds
0,42,0,10,0.9003,0.8926,517.9590
1,42,10,10,0.8622,0.8470,516.4771
2,42,20,10,0.8248,0.8071,516.3413
3,42,40,10,0.6322,0.5769,516.3254
4,43,0,10,0.9077,0.9039,516.2161
5,43,10,10,0.8602,0.8447,516.3628
6,43,20,10,0.8317,0.8057,516.4628
7,43,40,10,0.6788,0.6282,516.1788
8,44,0,10,0.9020,0.8928,516.0744
9,44,10,10,0.8520,0.8376,516.1069


In [13]:
REPORT_SEED = 42
REPORT_NOISE_PERCENT = 0
report_key = (REPORT_SEED, REPORT_NOISE_PERCENT)
print(f"Per-class validation scores: seed {REPORT_SEED}, noise {REPORT_NOISE_PERCENT}%")
per_class_validation_report = pd.DataFrame(sft_classification_reports[report_key]).T.loc[
    class_names.tolist(), ["precision", "recall", "f1-score", "support"]]
per_class_validation_report["support"] = per_class_validation_report["support"].astype(int)
display(per_class_validation_report.round(4))


Per-class validation scores: seed 42, noise 0%


,precision,recall,f1-score,support
Cardiovascular diseases,0.9142,0.8885,0.9012,780
Chronic kidney disease,0.8500,0.9358,0.8908,109
Chronic respiratory diseases,0.8760,0.8794,0.8777,257
Diabetes mellitus,0.9307,0.9227,0.9267,233
Digestive diseases,0.7607,0.7876,0.7739,113
HIV/AIDS,0.8727,0.9474,0.9085,152
Hepatitis A/B/C/E,0.9825,0.9180,0.9492,61
Mental disorders,0.9050,0.8926,0.8988,363
Musculoskeletal disorders,0.8742,0.9263,0.8995,285
Neoplasms (cancer),0.9342,0.9239,0.9291,815


# 8. Final evaluation on the held-out test set

Report each final-epoch run's accuracy, macro-F1, per-class precision/recall/F1 and support, training time, and peak **PyTorch-allocated GPU memory in GiB**. Training time includes the trainable encoder and head; no shared feature-extraction cost is allocated. Model initialization, copying final weights to CPU, and test evaluation are excluded from training_seconds.

Upload the existing **data/run_results.csv** from the repository to **/content/run_results.csv** first to retain the frozen results. This cell updates only full-fine-tuning rows and saves after each evaluated run. If no file exists, it creates the same 36-run registry. Download the updated CSV when finished. Seed averages, performance drops, and comparison charts belong in data_and_results.ipynb.


In [14]:
expected_run_keys = {(seed, noise) for seed in RANDOM_SEEDS for noise in NOISE_PERCENTAGES}
if "sft_validation_training_run_id" not in globals():
    raise RuntimeError("Run sections 5–7 before final test evaluation.")
assert sft_validation_training_run_id == sft_training_run_id, "Rerun section 7 for the current models."
assert sft_training_config == current_training_config(), "Training settings changed; rerun sections 6–7."
assert set(sft_model_states) == set(sft_run_metadata) == set(sft_validation_metadata) == expected_run_keys
validation_by_run = sft_validation_results.set_index(["seed", "noise_percent"])
assert validation_by_run.index.is_unique and set(validation_by_run.index) == expected_run_keys
assert validation_by_run["final_epoch"].eq(NUMBER_OF_EPOCHS).all()
check_source_data()

RESULTS_PATH = DATA_DIRECTORY / "run_results.csv"
if RESULTS_PATH.is_file():
    run_registry = pd.read_csv(RESULTS_PATH, dtype=str, keep_default_na=False)
else:
    print("No existing results CSV found: creating a new registry without previous frozen results. Upload the combined run_results.csv first to retain them.")
    run_registry = pd.DataFrame([
        {"run_id": f"{'lora_r8' if method == 'lora' else method}_seed{seed}_noise{noise:02d}",
         "model": method, "lora_rank": "8" if method == "lora" else "", "seed": str(seed),
         "requested_noise_percent": str(noise), "status": "planned"}
        for method in ("frozen", "lora", "full_finetuning")
        for seed in RANDOM_SEEDS for noise in NOISE_PERCENTAGES])
assert {"run_id", "model"}.issubset(run_registry.columns)
assert run_registry["run_id"].str.strip().ne("").all() and run_registry["run_id"].is_unique
run_registry = run_registry.set_index("run_id")
for seed, noise in expected_run_keys:
    run_id = f"full_finetuning_seed{seed}_noise{noise:02d}"
    if run_id in run_registry.index:
        assert run_registry.at[run_id, "model"] == "full_finetuning", run_id

sft_test_result_rows, sft_per_class_rows = [], []
sft_test_predictions = {}
for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        run_key = (seed, noise_percent)
        synchronize_gpu()
        reset_gpu_peak_memory()
        model, metadata = restore_full_model(run_key)
        validation_record = sft_validation_metadata[run_key]
        assert metadata["training_run_id"] == validation_record["training_run_id"] == sft_training_run_id, "Models changed since validation; rerun section 7."
        synchronize_gpu()
        test_start = time.perf_counter()
        test_metrics = evaluate_full_model(model, "test", true_test_labels)
        synchronize_gpu()
        test_seconds = time.perf_counter() - test_start
        sft_test_predictions[run_key] = test_metrics["predictions"]
        report = classification_report(true_test_labels, test_metrics["predictions"],
                                       labels=CLASS_IDS, output_dict=True, zero_division=0)
        class_scores = [
            {"class_id": int(class_id), "class_name": str(class_names.loc[class_id]),
             "precision": report[str(class_id)]["precision"], "recall": report[str(class_id)]["recall"],
             "f1": report[str(class_id)]["f1-score"], "support": int(report[str(class_id)]["support"])}
            for class_id in CLASS_IDS]
        sft_per_class_rows.extend({"seed": seed, "noise_percent": noise_percent, **row} for row in class_scores)
        validation_row = validation_by_run.loc[run_key]
        measurements = metadata["measurements"]
        result = {
            "run_id": f"full_finetuning_seed{seed}_noise{noise_percent:02d}",
            "model": "full_finetuning", "lora_rank": "", "seed": seed,
            "requested_noise_percent": noise_percent,
            "actual_noise_percent": 100 * np.mean(training_label_variants[run_key] != original_training_labels),
            "status": "completed", "final_epoch": metadata["epoch"],
            "validation_accuracy": validation_row["accuracy"], "validation_macro_f1": validation_row["macro_f1"],
            "test_accuracy": test_metrics["accuracy"], "test_macro_f1": test_metrics["macro_f1"],
            "training_seconds": measurements["training_seconds"],
            "training_and_validation_seconds": measurements["training_and_validation_seconds"],
            "epoch_validation_seconds": measurements["epoch_validation_seconds"],
            "final_validation_seconds": validation_record["seconds"], "test_inference_seconds": test_seconds,
            "peak_pytorch_allocated_gib": maximum_measured_memory(measurements["peak_pytorch_allocated_gib"],
                validation_record["peak_pytorch_allocated_gib"], peak_gpu_memory_gib()),
            "gpu_model": metadata["environment"]["gpu"], "trainable_parameters": metadata["trainable_parameters"],
            "bert_checkpoint": BERT_CHECKPOINT_NAME, "max_token_length": MAXIMUM_TOKEN_LENGTH,
            "learning_rate": LEARNING_RATE, "training_batch_size": TRAINING_BATCH_SIZE,
            "evaluation_batch_size": EVALUATION_BATCH_SIZE, "weight_decay": WEIGHT_DECAY,
            "stopping_rule": "final_epoch", "training_precision": "float32",
            "torch_version": metadata["environment"]["torch"], "transformers_version": metadata["environment"]["transformers"],
            **{f"{split}_csv_sha256": source_hashes[split] for split in SPLIT_NAMES},
            "test_per_class_metrics_json": json.dumps(class_scores, ensure_ascii=False, allow_nan=False),
        }
        sft_test_result_rows.append(result)
        run_id = result["run_id"]
        for column in result:
            if column != "run_id" and column not in run_registry:
                run_registry[column] = ""
        if run_id not in run_registry.index:
            run_registry.loc[run_id] = ""
        obsolete_columns = ["head_training_seconds", "test_head_inference_seconds",
                            "amortized_train_encoding_seconds", "amortized_validation_encoding_seconds", "amortized_test_encoding_seconds",
                            "validation_accuracy_drop_pp", "validation_macro_f1_drop_pp", "test_accuracy_drop_pp", "test_macro_f1_drop_pp"]
        for column in obsolete_columns:
            if column in run_registry:
                run_registry.at[run_id, column] = ""
        for column, value in result.items():
            if column != "run_id":
                run_registry.at[run_id, column] = "" if pd.isna(value) else str(value)
        temporary_results_path = RESULTS_PATH.with_suffix(".csv.tmp")
        run_registry.reset_index().to_csv(temporary_results_path, index=False)
        temporary_results_path.replace(RESULTS_PATH)
        print(f"Saved {run_id}.")
        del model
        gc.collect()
        if computation_device.type == "cuda":
            torch.cuda.empty_cache()

sft_final_results = pd.DataFrame(sft_test_result_rows)
sft_test_per_class = pd.DataFrame(sft_per_class_rows)
print(f"\nSaved all {len(sft_final_results)} full-fine-tuning runs to {RESULTS_PATH}.")
print("Final test metrics use a 0–1 scale:")
display(sft_final_results[["seed", "requested_noise_percent", "test_accuracy", "test_macro_f1",
                           "training_seconds", "peak_pytorch_allocated_gib"]].round(4))
for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        print(f"\nPer-class test scores: seed {seed}, noise {noise_percent}%")
        scores = sft_test_per_class.loc[sft_test_per_class["seed"].eq(seed) & sft_test_per_class["noise_percent"].eq(noise_percent),
                                       ["class_id", "class_name", "precision", "recall", "f1", "support"]]
        display(scores.reset_index(drop=True).round(4))


Saved full_finetuning_seed42_noise00.
Saved full_finetuning_seed42_noise10.
Saved full_finetuning_seed42_noise20.
Saved full_finetuning_seed42_noise40.
Saved full_finetuning_seed43_noise00.
Saved full_finetuning_seed43_noise10.
Saved full_finetuning_seed43_noise20.
Saved full_finetuning_seed43_noise40.
Saved full_finetuning_seed44_noise00.
Saved full_finetuning_seed44_noise10.
Saved full_finetuning_seed44_noise20.
Saved full_finetuning_seed44_noise40.

Saved all 12 full-fine-tuning runs to /content/run_results.csv.
Final test metrics use a 0–1 scale:


,seed,requested_noise_percent,test_accuracy,test_macro_f1,training_seconds,peak_pytorch_allocated_gib
0,42,0,0.9003,0.8925,517.9590,6.1720
1,42,10,0.8629,0.8501,516.4771,6.1713
2,42,20,0.8324,0.8167,516.3413,6.1713
3,42,40,0.6388,0.5867,516.3254,6.1713
4,43,0,0.9058,0.8992,516.2161,6.1713
5,43,10,0.8721,0.8556,516.3628,6.1713
6,43,20,0.8307,0.7992,516.4628,6.1713
7,43,40,0.6653,0.6116,516.1788,6.1713
8,44,0,0.9069,0.8976,516.0744,6.1713
9,44,10,0.8578,0.8331,516.1069,6.1713



Per-class test scores: seed 42, noise 0%


,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.9188,0.8835,0.9008,781
1,1,Chronic kidney disease,0.8547,0.9174,0.8850,109
2,2,Chronic respiratory diseases,0.8346,0.8833,0.8582,257
3,3,Diabetes mellitus,0.9356,0.9356,0.9356,233
4,4,Digestive diseases,0.7797,0.8364,0.8070,110
5,5,HIV/AIDS,0.8788,0.9539,0.9148,152
6,6,Hepatitis A/B/C/E,0.8750,0.9333,0.9032,60
7,7,Mental disorders,0.8753,0.9041,0.8895,365
8,8,Musculoskeletal disorders,0.9266,0.9233,0.9250,287
9,9,Neoplasms (cancer),0.9207,0.9207,0.9207,820



Per-class test scores: seed 42, noise 10%


...

,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.9306,0.7900,0.8546,781
1,1,Chronic kidney disease,0.6963,0.8624,0.7705,109
2,2,Chronic respiratory diseases,0.8367,0.7977,0.8167,257
3,3,Diabetes mellitus,0.9401,0.8755,0.9067,233
4,4,Digestive diseases,0.6538,0.7727,0.7083,110
5,5,HIV/AIDS,0.8034,0.9408,0.8667,152
6,6,Hepatitis A/B/C/E,0.6404,0.9500,0.7651,60
7,7,Mental disorders,0.8295,0.8795,0.8537,365
8,8,Musculoskeletal disorders,0.9088,0.8676,0.8877,287
9,9,Neoplasms (cancer),0.8929,0.9146,0.9036,820



Per-class test scores: seed 44, noise 20%


,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.8327,0.8668,0.8494,781
1,1,Chronic kidney disease,0.7739,0.8165,0.7946,109
2,2,Chronic respiratory diseases,0.8734,0.8054,0.8381,257
3,3,Diabetes mellitus,0.8843,0.8197,0.8508,233
4,4,Digestive diseases,0.8158,0.5636,0.6667,110
5,5,HIV/AIDS,0.7784,0.9013,0.8354,152
6,6,Hepatitis A/B/C/E,0.6795,0.8833,0.7681,60
7,7,Mental disorders,0.8415,0.8438,0.8427,365
8,8,Musculoskeletal disorders,0.7409,0.9268,0.8235,287
9,9,Neoplasms (cancer),0.9023,0.8902,0.8963,820



Per-class test scores: seed 44, noise 40%


,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.8569,0.6287,0.7253,781
1,1,Chronic kidney disease,0.2551,0.8073,0.3877,109
2,2,Chronic respiratory diseases,0.5445,0.6187,0.5792,257
3,3,Diabetes mellitus,0.6404,0.7339,0.6840,233
4,4,Digestive diseases,0.3878,0.5182,0.4436,110
5,5,HIV/AIDS,0.5949,0.7632,0.6686,152
6,6,Hepatitis A/B/C/E,0.4095,0.7167,0.5212,60
7,7,Mental disorders,0.7500,0.6822,0.7145,365
8,8,Musculoskeletal disorders,0.7614,0.7003,0.7296,287
9,9,Neoplasms (cancer),0.8440,0.7256,0.7803,820
